### packages

In [ ]:
import pandas as pd

### read data

In [ ]:
netzero = pd.read_excel("data/netzero_2026.xlsx")
forbes = pd.read_csv("data/Forbes_2000_Companies_2026.csv")

In [ ]:
#view companies
#display(netzero[netzero["Entity_type"] == "Company"])

#forbes

### merge

In [ ]:
#merge forbes data onto netzero data

#first make a temporary key column (company name) in forbes with the same name as in netzero 
forbes["Name"] = forbes["Company"]

#merge
merged = pd.merge(
    netzero, 
    forbes[["Name", "Sales ($B)", "Profit ($B)"]], #only need sales and profit columns
    on = "Name",
    how = "left")

In [ ]:
#verify merge worked by viewing some important columns from both dfs
merged[["Name", "End_target", "Sales ($B)", "Profit ($B)"]]

,Name,End_target,Sales ($B),Profit ($B)
0,Afghanistan,Reduction v. BAU,NaN,NaN
1,Angola,Emissions reduction target,NaN,NaN
2,Argentina,Net zero,NaN,NaN
3,Armenia,Climate neutral,NaN,NaN
4,Antigua and Barbuda,Net zero,NaN,NaN
...,...,...,...,...
4175,HP Inc.,Net zero,NaN,NaN
4176,TAQA,Net zero,14.97,2.03
4177,Blackstone,Other,14.88,3.05
4178,GE Vernova,Net zero,39.35,9.38


#### identify overlap

In [ ]:
#find the number of Forbes 2000 companies not included in the NetZero data

#define companies in each dataset
forbes_comps = forbes[["Company"]].copy() #df column of 2000 companies in Forbes
netzero_comps = (merged[merged["Entity_type"] == "Company"])["Name"].copy().unique() #list of companies in NetZero data


#new column that evaluates whether each Forbes company is also in NetZero
forbes_comps["in_netzero"] = forbes_comps["Company"].isin(netzero_comps)

#print number of Falses (number of Forbes companies not in NetZero)
2000 - forbes_comps["in_netzero"].sum()

880

In [78]:
#view excluded companies
excluded_comps = forbes_comps[forbes_comps["in_netzero"] == False]
excluded_comps

,Company,in_netzero
0,JPMorganChase,False
4,Saudi Arabian Oil Company (Saudi Aramco),False
12,HSBC Holdings,False
13,ExxonMobil,False
14,Samsung Electronics,False
...,...,...
1994,Kuwait Projects,False
1995,Zscaler,False
1996,Federal Bank,False
1997,Graubunder Ktbk,False


Looking for inconsistencies, I saw that JP Morgan Chase *is* in both datasets, but it is formatted without spaces in Forbes. I counted how many entities are affected by simply a lack of spaces (see below), but there were only two. There's also an issue with one dataset including business suffixes (e.g., "Co") the other did not.

In [ ]:
#reformat NetZero company names
netzero_reform = []
for comp in netzero_comps:
    netzero_reform.append(comp.replace(" ", "")) #replace spaces with nothing

#return companies that found a match after removing spaces
excluded_comps[excluded_comps["Company"].isin(netzero_reform) == True]

,Company,in_netzero
0,JPMorganChase,False
13,ExxonMobil,False
